# Restructuration

Une fois toutes les données téléchargées, certaines restructurations sont nécessaires. 

Voir le [README](./README.md) pour plus de détails.

In [48]:
import pandas as pd

# GCPE

### GCPE IFT : hétérogénéïté entre 2017 et 2021

Le fichier obtenu pour l'enquête PK 2017 expose uniquement la colonne `Orge` alors que celui de l'enquête PK 2021 expose à la fois `Orge de printemps` et `Orge d'hiver`.

Pour fusionner `Orge de printemps` et `Orge d'hiver` de façon cohérente, on a besoin de la surface développée de chacune de ces cultures en 2021.

On modifie donc ici : 
- `ift_culture_ancienne_region_gcpe_2021.csv` -> fusion des colonnes d'orge. 
- `surface_espece_ancienne_region.csv` -> fusion des colonnes d'orge.

> Attention, pour certaines lignes, on a pas de valeur d'IFT pour l'Orge de printemps. Dans ces cas là, on considéreras que l'IFT Orge total correspondra à l'IFT Orge d'hiver.

In [49]:
# chargement des données
df = {}
df['ift_culture_ancienne_region_gcpe_2017'] = pd.read_csv('./ift/gcpe/ift_culture_ancienne_region_gcpe_2017.csv')
df['ift_culture_ancienne_region_gcpe_2021'] = pd.read_csv('./ift/gcpe/ift_culture_ancienne_region_gcpe_2021.csv')
df['surface_espece_ancienne_region'] = pd.read_csv('./surface/gcpe/surface_espece_ancienne_region.csv')

In [50]:
df['surface_espece_ancienne_region_2021_orge_printemps'] = df['surface_espece_ancienne_region'].loc[
    (df['surface_espece_ancienne_region']['Campagne'] == 2021) &
    (df['surface_espece_ancienne_region']['Espece_SSP'] == 'Orge de printemps')
].set_index('Espece_SSP')[['Nom_Ancienne_Region', 'Surface_Espece_Region']]

df['surface_espece_ancienne_region_2021_orge_hiver'] = df['surface_espece_ancienne_region'].loc[
    (df['surface_espece_ancienne_region']['Campagne'] == 2021) &
    (df['surface_espece_ancienne_region']['Espece_SSP'] == "Orge d'hiver")
].set_index('Espece_SSP')[['Nom_Ancienne_Region', 'Surface_Espece_Region']]

# ajout de la surface de l'orge d'hiver
left = df['ift_culture_ancienne_region_gcpe_2021'] 
right = df['surface_espece_ancienne_region_2021_orge_hiver'].rename(columns={
    'Surface_Espece_Region' : 'surface_orge_hiver'
})
df['ift_culture_ancienne_region_gcpe_2021_extanded'] = pd.merge(
    left, 
    right, 
    left_on = ['nom_ancienne_region'], 
    right_on = ['Nom_Ancienne_Region']
)

# ajout de la surface de l'orge de printemps
left = df['ift_culture_ancienne_region_gcpe_2021_extanded'] 
right = df['surface_espece_ancienne_region_2021_orge_printemps'].rename(columns={
    'Surface_Espece_Region' : 'surface_orge_printemps'
})
df['ift_culture_ancienne_region_gcpe_2021_extanded'] = pd.merge(
    left, 
    right, 
    left_on = ['nom_ancienne_region'], 
    right_on = ['Nom_Ancienne_Region']
)

# calcul surface totale orge
df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_hiver'] + \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_printemps']

# identification des cas où on a pas d'ift orge printemps
region_sans_orge_printemps = df['ift_culture_ancienne_region_gcpe_2021_extanded'].loc[
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['Orge de printemps'].isna()
].index

# calcul des proportions
df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_hiver'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_hiver'] / \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge']


df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_printemps'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_printemps'] / \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge']

# calcul de l'IFT orge total
df['ift_culture_ancienne_region_gcpe_2021_extanded']['Orge'] = \
    round(df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_printemps']*df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge de printemps"] + \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_hiver']*df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge d'hiver"], 2)

# correction dans le cas où IFT orge printemps non accessible
df['ift_culture_ancienne_region_gcpe_2021_extanded'].loc[
    region_sans_orge_printemps
    , 'Orge'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge d'hiver"]

# correction de surface_espece_ancienne_region (fusion de Orge de printemps et Orge)
df['surface_espece_ancienne_region']['Espece_SSP_corrige'] = \
    df['surface_espece_ancienne_region']['Espece_SSP']

df['surface_espece_ancienne_region'].loc[
    df['surface_espece_ancienne_region']['Espece_SSP_corrige'].isin([
        "Orge de printemps", "Orge d'hiver"
    ]), 'Espece_SSP_corrige'
] = "Orge"

df['surface_espece_ancienne_region_corrige'] = df['surface_espece_ancienne_region'].groupby([
    "Espece_SSP_corrige", "Campagne", "Nom_Ancienne_Region"
]).agg({
    'Surface_Espece_Region' : 'sum',
    'Surface_Region' : 'first', 
    'Part_surface_espece_region' : 'sum'
}).reset_index().sort_values([
    'Nom_Ancienne_Region', 'Campagne', 'Espece_SSP_corrige'
])

# export des informations
df['surface_espece_ancienne_region_corrige'].to_csv('./restructure/surface_espece_ancienne_region_restructure.csv', index=False)
df['ift_culture_ancienne_region_gcpe_2021_extanded'][
    list(df['ift_culture_ancienne_region_gcpe_2021'].columns) + ['Orge']
].to_csv('./restructure/ift_culture_ancienne_region_gcpe_2021_restructure.csv', index=False)

# Viticulture

### Viticulture ift : hétérogénité des bassins viticoles disponibles entre 2016, 2019 et 2024

<table>
  <thead>
    <tr>
      <th>Basins viticoles</th>
      <th>2016</th>
      <th>2019</th>
      <th>2024</th>
    </tr>
  </thead>
  <tbody>
    <!-- Rows with all “Oui” -->
    <tr style="background-color:#d8f8d8;">
      <td>Alsace</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Beaujolais</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Bordelais</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Champagne</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Cher</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Corse</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Gaillac</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Gers</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Val de Loire</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr style="background-color:#d8f8d8;">
      <td>Pyrénées‑Orientales</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Bugey‑Savoie</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Bugey Savoie hors vallée du Rhône</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Charentes</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Charentes‑Cognac</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Côtes‑du‑Rhône Nord</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Côtes‑du‑Rhône Sud</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Vallée du Rhône Nord</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Vallée du Rhône Sud</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Dordogne</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Dordogne‑Duras</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Jura</td>
      <td>Non</td>
      <td>Oui</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Languedoc hors Pyrénées‑Orientales</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Languedoc hors vallée du Rhône</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Lot‑et‑Garonne</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Lot‑et‑Garonne hors Duras</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
    <tr>
      <td>Provence (Var‑Vaucluse)</td>
      <td>Oui</td>
      <td>Oui</td>
      <td>Non</td>
    </tr>
    <tr>
      <td>Provence‑Méditerranée</td>
      <td>Non</td>
      <td>Non</td>
      <td>Oui</td>
    </tr>
  </tbody>
</table>

Les restructurations proposées sont donc les suivantes : 
- Bugey-Savoie = Bugey Savoie hors vallée du Rhône
- Charentes = Charentes-Cognac
- Côtes-du-Rhône Sud = Vallée du Rhône Sud
- Côtes-du-Rhône Nord = Vallée du Rhône Nord
- Dordogne = Dordogne-Duras
- Lot-et-Garonne hors Duras = Lot-et-Garonne
- Provence (Var-Vaucluse) = Provence-Méditerranée
- Languedoc hors Pyrénées-Orientales + Pyrénées Orientales = Languedoc hors valée du Rhône + Côtes-du-Rhône Sud = Languedoc

In [55]:
# chargement des données
df = {}
df['ift_culture_bassin_viticulture_2016'] = pd.read_csv('./ift/viticulture/ift_culture_bassin_viticulture_2016.csv')
df['ift_culture_bassin_viticulture_2019'] = pd.read_csv('./ift/viticulture/ift_culture_bassin_viticulture_2019.csv')
df['ift_culture_bassin_viticulture_2024'] = pd.read_csv('./ift/viticulture/ift_culture_bassin_viticulture_2024.csv')

df['matching_bassin_2016_2019_to_2024'] =  pd.DataFrame({
    '2016_2019' : ['Bugey-Savoie', 'Charentes', 'Côtes-du-Rhône Sud', 'Côtes-du-Rhône Nord', 'Dordogne', 'Lot-et-Garonne hors Duras', 'Provence (Var-Vaucluse)'],
    '2024' : ['Bugey Savoie hors vallée du Rhône', 'Charentes-Cognac', 'Vallée du Rhône Sud', 'Vallée du Rhône Nord', 'Dordogne-Duras', 'Lot-et-Garonne', 'Provence-Méditerranée']
})

In [56]:
df['ift_culture_bassin_viticulture_2016']

,nom_bassin_viticole,ift_total
0,Alsace,14.9
1,Beaujolais,18.7
2,Bordelais,17.2
3,Bouches-du-Rhône,9.3
4,Bourgogne,19.3
5,Bugey-Savoie,18.1
6,Cahors,15.8
7,Champagne,23.5
8,Charentes,18.0
9,Cher,17.6


### Matching simple : on prend les conventions de 2024

In [57]:
# dans 2016, on remplace par les labels de bassins de 2024
left = df['ift_culture_bassin_viticulture_2016']
right = df['matching_bassin_2016_2019_to_2024']
df['ift_culture_bassin_viticulture_2016_match'] = pd.merge(left, right, left_on = 'nom_bassin_viticole', right_on='2016_2019', how='left')
df['ift_culture_bassin_viticulture_2016_match']['2024'] = \
    df['ift_culture_bassin_viticulture_2016_match']['2024'].fillna(df['ift_culture_bassin_viticulture_2016_match']['nom_bassin_viticole'])

df['ift_culture_bassin_viticulture_2016_match'] = df['ift_culture_bassin_viticulture_2016_match'][['2024', 'ift_total']].rename(columns={
    '2024' : 'nom_bassin_viticole'
})

# dans 2019, on remplace par les labels de bassins de 2024
left = df['ift_culture_bassin_viticulture_2019']
right = df['matching_bassin_2016_2019_to_2024']
df['ift_culture_bassin_viticulture_2019_match'] = pd.merge(left, right, left_on = 'nom_bassin_viticole', right_on='2016_2019', how='left')
df['ift_culture_bassin_viticulture_2019_match']['2024'] = \
    df['ift_culture_bassin_viticulture_2019_match']['2024'].fillna(df['ift_culture_bassin_viticulture_2019_match']['nom_bassin_viticole'])

df['ift_culture_bassin_viticulture_2019_match'] = df['ift_culture_bassin_viticulture_2019_match'][['2024', 'ift_total']].rename(columns={
    '2024' : 'nom_bassin_viticole'
})

#### Cas complexes : on doit modifier des deux côtés (faire une somme)

1. Languedoc hors Pyrénées-Orientales + Pyrénées Orientales = Languedoc hors valée du Rhône + Côtes-du-Rhône Sud = Languedoc

In [58]:
df['ift_culture_bassin_viticulture_2016_match'].loc[
    df['ift_culture_bassin_viticulture_2016_match']['nom_bassin_viticole'].isin([
        'Languedoc hors Pyrénées-Orientales', 'Pyrénées-Orientales'
    ])
]

,nom_bassin_viticole,ift_total
16,Languedoc hors Pyrénées-Orientales,14.0
19,Pyrénées-Orientales,10.4


In [59]:
df['ift_culture_bassin_viticulture_2019_match'].loc[
    df['ift_culture_bassin_viticulture_2019_match']['nom_bassin_viticole'].isin([
        'Languedoc hors Pyrénées-Orientales', 'Pyrénées-Orientales'
    ])
]

,nom_bassin_viticole,ift_total
17,Languedoc hors Pyrénées-Orientales,11.1
20,Pyrénées-Orientales,7.9


In [60]:
df['ift_culture_bassin_viticulture_2024'].loc[
    df['ift_culture_bassin_viticulture_2024']['nom_bassin_viticole'].isin([
        'Languedoc hors vallée du Rhône', 'Vallée du Rhône Sud'
    ])
]

,nom_bassin_viticole,ift_total
14,Languedoc hors vallée du Rhône,11.3
20,Vallée du Rhône Sud,11.4


In [61]:
df['ift_culture_bassin_viticulture_2024']

,nom_bassin_viticole,ift_total
0,Alsace,13.9
1,Beaujolais,20.0
2,Bordelais,19.2
3,Bourgogne,21.2
4,Bugey Savoie hors vallée du Rhône,16.0
5,Cahors,18.1
6,Champagne,26.2
7,Charentes-Cognac,17.1
8,Cher,21.0
9,Corse,10.7


-